In [1]:
import sys
from pathlib import Path

try:
    # 일반 Python 스크립트 실행 시 (__file__이 존재)
    current_path = Path(__file__).resolve()
except NameError:
    # Jupyter Notebook 실행 시 (__file__ 없음)
    current_path = Path().resolve()

# 현재 경로에서 stock_forecast 폴더까지 자동 탐색
for parent in current_path.parents:
    if (parent / "stock_forecast" / "DATA").is_dir():
        stock_forecast_path = parent / "stock_forecast"
        break
else:
    raise ImportError("stock_forecast/DATA 폴더를 찾을 수 없습니다.")

# sys.path에 추가
if str(stock_forecast_path) not in sys.path:
    sys.path.insert(0, str(stock_forecast_path))

print(f"sys.path에 등록된 경로: {stock_forecast_path}")

sys.path에 등록된 경로: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\stock_forecast


In [2]:
import pandas as pd
from sqlalchemy import create_engine
from DATA.stock_invest_function import *


In [3]:
def add_yoy_growth(df, value_column='value', group_column='root_hs_code', date_column='date'):
    """
    root_hs_code별로 value 컬럼의 연간 증가율을 계산하여 새로운 컬럼으로 추가합니다.
    """
    df = df.copy()
    # top_company_codes = avg_yoy_by_code.sort_values(by='avg_forecast_yoy', ascending=False).head(company_num)
    df = df.dropna(axis=0)
    df[date_column] = pd.to_datetime(df[date_column])
    df.sort_values(by=[group_column, date_column], inplace=True)

    # YoY (12개월 전 대비 비율 변화율) 계산
    df[f'{value_column}_yoy'] = (
        df.groupby(group_column)[value_column]
        .transform(lambda x: x.pct_change(periods=12))
    )

    return df

In [4]:
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host' : '192.168.0.230',
    'host': get_db_host(),         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

trade_df = fetch_table_data(db_info, 'korea_monthly_trade_data_forecast')

import pandas as pd

# date를 datetime으로 변환
trade_df['date'] = pd.to_datetime(trade_df['date'])

# 결측치 제거
trade_df = trade_df.dropna(subset=['expDlr_forecast_12m'])

# root_hs_code, date로 정렬
trade_df = trade_df.sort_values(['root_hs_code', 'date'])

# 그룹 연산 준비
grouped = trade_df.groupby('root_hs_code')

# 이전 12개월 합계 (trailing)
trade_df['export_trail_12m'] = grouped['expDlr_forecast_12m'].transform(
    lambda x: x.rolling(window=12, min_periods=12).sum()
)

# 이후 12개월 합계 (forward)
# shift(-11)은 앞으로 11개월 밀어 rolling 12로 보면 해당 시점 기준 이후 12개월을 의미
trade_df['export_forward_12m'] = grouped['expDlr_forecast_12m'].transform(
    lambda x: x.shift(-11).rolling(window=12, min_periods=12).sum()
)

# YoY 성장률
trade_df['export_yoy_growth'] = (
    (trade_df['export_forward_12m'] / trade_df['export_trail_12m']) - 1
)

# 필요한 컬럼만 보기
result_df = trade_df[['date', 'root_hs_code',
                      'export_trail_12m', 'export_forward_12m', 'export_yoy_growth']]

# 필요시 최근 데이터만
trade_yoy_growth = result_df[result_df['date'] == '2025-06-30']

# 확인
# print(trade_yoy_growth.head(20))


✅ 'korea_monthly_trade_data_forecast' 테이블에서 241385건의 데이터를 가져왔습니다.


In [5]:
len(trade_yoy_growth['root_hs_code'].unique().tolist())

1020

In [6]:
# ticker  ='A353200'
# hscd = ''

In [7]:
trade_yoy_growth[trade_yoy_growth['root_hs_code'] == '854232']

,date,root_hs_code,export_trail_12m,export_forward_12m,export_yoy_growth
221625,2025-06-30,854232,7.638940e+10,9.837231e+10,0.287774


In [8]:
company_df = fetch_table_data(db_info, 'korea_company_hscode_map')
company_df = company_df.rename(columns={'hs_code' : 'root_hs_code'})
# company_df.rename(columns={'hs_code_6d': 'root_hs_code'}, inplace=True)

✅ 'korea_company_hscode_map' 테이블에서 762건의 데이터를 가져왔습니다.


In [9]:
company_df['root_hs_code'] = company_df['root_hs_code'].astype(str)
trade_yoy_growth['root_hs_code'] = trade_yoy_growth['root_hs_code'].astype(str)

monster_df = pd.merge(company_df, trade_yoy_growth, on='root_hs_code', how='left', indicator=True)

In [14]:
monster_df[monster_df['Name'] == '메타바이오메드']

,ticker,Name,root_hs_code,date,export_trail_12m,export_forward_12m,export_yoy_growth,_merge
608,A059210,메타바이오메드,9018498000,2025-06-30,213665200.0,211773400.0,-0.008854,both
609,A059210,메타바이오메드,9018499000,2025-06-30,34014050.0,33580600.0,-0.012743,both
610,A059210,메타바이오메드,3006402000,2025-06-30,78180740.0,84384120.0,0.079347,both


In [12]:
import os
from datetime import date

# 저장 경로
path = r'C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results'

# 오늘 날짜
today = date.today().strftime("%Y-%m-%d")

# 파일명
file_name = f"main_products_export_results_{today}.xlsx"
file_path = os.path.join(path, file_name)

# 엑셀 저장
monster_df.to_excel(file_path, index=False)

print(f"저장 완료: {file_path}")

저장 완료: C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\main_products_export_results_2025-12-14.xlsx


In [13]:
trade_df.tail(12)

,date,root_hs_code,final_expDlr_yoy,expDlr_forecast_12m,export_trail_12m,export_forward_12m,export_yoy_growth
211553,2024-08-31,970191,0.962055,12960500.0,259466190.0,217234500.0,-0.162764
212574,2024-09-30,970191,-0.108287,35002100.0,255215690.0,NaN,NaN
213593,2024-10-31,970191,-0.188063,38237000.0,246359190.0,NaN,NaN
214616,2024-11-30,970191,-0.659619,8015870.0,230825360.0,NaN,NaN
215639,2024-12-31,970191,-0.402585,9780640.0,224234500.0,NaN,NaN
216661,2025-01-31,970191,-0.364995,11238300.0,217774800.0,NaN,NaN
217682,2025-02-28,970191,-0.300595,13075200.0,212155100.0,NaN,NaN
218704,2025-03-31,970191,-0.092616,13468200.0,210780400.0,NaN,NaN
219729,2025-04-30,970191,-0.077535,37093900.0,207662600.0,NaN,NaN
220752,2025-05-31,970191,0.359536,19823800.0,212905100.0,NaN,NaN
